# 🔁 Main Pipeline – Jupyter Notebook Version
Notebook ini merupakan versi interaktif dari `main.py`, dan digunakan untuk menjalankan pipeline data mining end-to-end.

In [ ]:
# Setup: Menambahkan path dan import library
import sys
sys.path.append("../src")

import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Import fungsi-fungsi dari modul src/
from preprocessing import preprocess_pipeline
from model import split_data_timeseries, train_model, evaluate_model, get_feature_importance
from utils import print_regression_report, plot_regression_results

In [ ]:
# STEP 1: LOAD PROCESSED DATA
print("="*60)
print("STEP 1: LOAD PROCESSED DATA")
print("="*60)

# Load data yang sudah diproses dari preprocessing_template.ipynb
processed_data_path = "../data/processed/data_preprocessing_encoded.csv"

if os.path.exists(processed_data_path):
    df = pd.read_csv(processed_data_path)
    print(f"✓ Data loaded! Shape: {df.shape}")
    print(f"\nKolom tersedia ({len(df.columns)}):")
    print(df.columns.tolist())
else:
    print(f"❌ Error: {processed_data_path} tidak ditemukan!")
    print("📋 Silakan jalankan notebook/preprocessing_template.ipynb terlebih dahulu")
    
df.head()

In [ ]:
# STEP 2: PREPARE FEATURES
print("\n" + "="*60)
print("STEP 2: PREPARE FEATURES")
print("="*60)

# Cari kolom target
target_col = None
if "Target_Harga_Bulan_Berikutnya" in df.columns:
    target_col = "Target_Harga_Bulan_Berikutnya"
elif "target" in df.columns:
    target_col = "target"

if target_col is None:
    print("❌ Kolom target tidak ditemukan!")
    print(f"Kolom yang ada: {df.columns.tolist()}")
else:
    print(f"✓ Target column: {target_col}")
    
    # Exclude kolom non-fitur
    exclude_cols = [target_col, "Komoditas", "Tahun", "Bulan", "Tanggal", "Harga", "Outlier"]
    feature_cols = [col for col in df.columns if col not in exclude_cols]
    
    print(f"✓ Total fitur: {len(feature_cols)}")
    print(f"  Contoh fitur: {feature_cols[:5]}...")

In [ ]:
# STEP 3: TIME-BASED DATA SPLIT
print("\n" + "="*60)
print("STEP 3: TIME-BASED DATA SPLIT (80% train, 20% test)")
print("="*60)

# Drop baris dengan missing values
df_clean = df.dropna(subset=[target_col]).copy()
print(f"Data setelah drop missing: {df_clean.shape}")

# Siapkan data untuk split
data_for_split = df_clean[feature_cols + [target_col] + 
                          (["Tanggal"] if "Tanggal" in df_clean.columns else [])].copy()

# Split dengan time-based split (PENTING untuk time series!)
date_col = "Tanggal" if "Tanggal" in data_for_split.columns else None
X_train, X_test, y_train, y_test = split_data_timeseries(
    data_for_split,
    target_column=target_col,
    test_size=0.2,
    date_column=date_col
)

print(f"✓ Time-based split completed")
print(f"  X_train: {X_train.shape}")
print(f"  X_test:  {X_test.shape}")
print(f"  y_train: {y_train.shape}")
print(f"  y_test:  {y_test.shape}")

In [ ]:
# STEP 4: TRAIN MODEL
print("\n" + "="*60)
print("STEP 4: TRAINING MODEL")
print("="*60)

model = train_model(
    X_train,
    y_train,
    n_estimators=100,
    max_depth=15,
    random_state=42
)
print("✓ Model berhasil dilatih dengan 100 trees dan max_depth=15")

In [ ]:
# STEP 6: VISUALIZATION & FEATURE IMPORTANCE
print("\n" + "="*60)
print("STEP 6: VISUALIZATION & FEATURE IMPORTANCE")
print("="*60)

# Actual vs Predicted Plot
plot_regression_results(y_test.values, y_pred)

# Feature Importance
print("\nTop 10 Feature Importance:")
feature_importance = get_feature_importance(model, X_test.columns, top_n=10)
print(feature_importance)

In [ ]:
# STEP 5: EVALUATE MODEL
print("\n" + "="*60)
print("STEP 5: MODEL EVALUATION")
print("="*60)

results = evaluate_model(model, X_test, y_test)
y_pred = results["y_pred"]

# Print metrik regresi
print_regression_report(y_test, y_pred)